# Õppetund 03 - Agentseid disainimustreid

Selles õppetunnis uurime kolme põhilist disainimustrit tõhusate AI-agentide loomiseks:

1. **Selged agendi juhised** — täpsete, rolli määratlevate käskluste koostamine agendi käitumise juhendamiseks
2. **Struktureeritud väljund Pydanticu mudelitega** — tagades, et agendid annavad etteaimatavat, valideeritud andmeid
3. **Ühe vastutusega agendid** — keskendunud agentide kujundamine, mis teevad igaüks ühte asja hästi

Rakendame iga mustrit **reisisihtkoha soovitaja** stsenaariumis, ehitades järk-järgult süsteemi, mis suudab soovitada sihtkohti, kontrollida saadavust ja hallata logistikat.


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Muster 1: Selged agendi juhised

Mõjukaim muster on ka lihtsaim: kirjutada agendile selged ja üksikasjalikud juhised.

Head juhised määratlevad:
- **Kes** agent on (persona ja toon)
- **Mida** ta peaks tegema (samm-sammult ülesanded)
- **Kuidas** ta peaks käituma (piirangud ja stiil)

Allpool loome reisikonserdi agendi, kellel on selged juhised, mis kujundavad iga tema antud vastuse.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Muster 2: Struktureeritud väljund Pydantic mudelitega

Vaba vormi tekst on vestluseks kasulik, kuid alluvatel süsteemidel on vaja struktureeritud andmeid.
Paaritades **Pydantic mudelid** koos **tööriistafunktsiooniga**, saame:

- Määratleda täpne skeem agendi väljundi jaoks
- Vastuseid automaatselt valideerida
- Integreerida agendi tulemused rakenduse loogikasse usaldusväärselt

Tähtis on `response_format` edasiandmine, kui agenti käivitame. See sunnib
mudelit tagastama valideeritud `TravelRecommendations` objekti (saadaval `response.value`)
asemel, et vaba vormi teksti. `get_destination_details` tööriist tagastab samuti tüübistatult
`DestinationRecommendation`, nii et andmed jäävad terve protsessi jooksul struktureerituks.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Muster 3: Ühe vastutuseagentuurid

Komplekssed ülesanded saavad kasu töölükkamisest mitme keskendunud agente vahel, millest igaühel on üks vastutusala:

- **Sihtkohaekspert**, kes tunneb kohti ja saadavust
- **Logistikaplaneerija**, kes tegeleb lendude, hotellide ja marsruutidega

See peegeldab tarkvaraarenduse põhimõtet *huvide eraldamisest* — iga agent on lihtsam testida, hooldada ja iseseisvalt täiustada.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Kokkuvõte

Selles õppetükis rakendasime kolme agentide kujundusmustrit reisisoovituste stsenaariumil:

| Muster | Peamine idee | Kasu |
|---|---|---|
| **Selged juhised** | Määratle persona, vastutused ja piirangud ette | Järjepidev, brändile vastav agendi käitumine |
| **Struktureeritud väljund** | Kasuta vastuse formaadina Pydantic mudelid | Kontrollitud, masinloetav tulemus |
| **Üks vastutusala** | Anna igale agendile üks kindel ülesanne | Lihtsam testida, hooldada ja kombineerida |

Need mustrid sobivad loomulikult kokku — saad kombineerida selged juhised struktureeritud väljundiga ühe vastutusalaga agendi sees, et luua vastupidavaid, tootmiskõlblikke süsteeme.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
